In [18]:
!pip install folium


In [19]:
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/Deforestation_Predictions_kerela.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/kerelakerela_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


In [21]:
# Function to compute thresholds
def get_thresholds(series):
    mean = series.mean()
    std = series.std()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# Compute thresholds for each index
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI thresholds:", ndvi_thresh)
print("NBR thresholds:", nbr_thresh)
print("BSI thresholds:", bsi_thresh)
print("NDMI thresholds:", ndmi_thresh)
df_merged = pd.merge(df_deforest, df_change[['latitude','longitude','NDVI_change','NBR_change','BSI_change','NDMI_change']],
                     on=['latitude','longitude'], how='left')
def classify_cause_adaptive(row):
    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Fire: large drop in NBR beyond 2 std deviations
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"
    # Logging: NDVI drop below 1st quartile + BSI increase above 3rd quartile
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"
    # Agriculture: moderate NDVI drop between q1 and mean, small BSI increase
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"
    else:
        return "Urban/Other"

# Apply classification
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)
df_merged.to_csv('/content/drive/MyDrive/Deforestation_Causes_Adaptive_kerela.csv', index=False)

# Summary
print("Deforestation Cause Summary (Adaptive Thresholds):")
print(df_merged['cause'].value_counts())


NDVI thresholds: {'mean': np.float64(-0.31016556737766665), 'std': 0.04901325486928982, 'q1': np.float64(-0.3461895725), 'q3': np.float64(-0.2798189), 'lower_2std': np.float64(-0.4081920771162463), 'upper_2std': np.float64(-0.21213905763908703)}
NBR thresholds: {'mean': np.float64(-0.2077333174310267), 'std': 0.05599800330357571, 'q1': np.float64(-0.24669844999999996), 'q3': np.float64(-0.17207930500000002), 'lower_2std': np.float64(-0.3197293240381781), 'upper_2std': np.float64(-0.09573731082387527)}
BSI thresholds: {'mean': np.float64(0.1162338241847185), 'std': 0.04815422087272139, 'q1': np.float64(0.08661235908281552), 'q3': np.float64(0.14825164853656603), 'lower_2std': np.float64(0.01992538243927572), 'upper_2std': np.float64(0.2125422659301613)}
NDMI thresholds: {'mean': np.float64(-0.09280389551820832), 'std': 0.04493088919446433, 'q1': np.float64(-0.12134231749999998), 'q3': np.float64(-0.06568500500000002), 'lower_2std': np.float64(-0.18266567390713698), 'upper_2std': np.floa

In [22]:
df_merged.to_csv('/content/drive/MyDrive/Deforestation_Causes_Adaptive_kerela.csv', index=False)

# Summary
print("Deforestation Cause Summary (Adaptive Thresholds):")
print(df_merged['cause'].value_counts())
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Load improved cause CSV
df = pd.read_csv('/content/drive/MyDrive/Deforestation_With_Cause_Improved.csv')

print(df['cause'].value_counts())
# Tamil Nadu roughly: latitude 8–13, longitude 76–80
m = folium.Map(location=[11.0, 78.5], zoom_start=7)
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)



Deforestation Cause Summary (Adaptive Thresholds):
cause
Urban/Other    3132
Agriculture    2206
Logging         597
Fire             43
Name: count, dtype: int64
cause
Urban/Other    687
Fire           547
Agriculture     13
Logging          8
Name: count, dtype: int64


In [23]:
import pandas as pd
import numpy as np

# Load adaptive cause CSV
df = pd.read_csv('/content/drive/MyDrive/Deforestation_Causes_Adaptive_kerela.csv')

# Check data
print(df.head())


   latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  9.217164  76.783948              1    -0.136629   -0.069963   -0.023063   
1  9.217164  76.804609              1    -0.220431   -0.169956    0.083165   
2  9.217164  76.820779              1    -0.285544   -0.184717    0.115918   
3  9.217164  76.828864              1    -0.269957   -0.193263    0.118349   
4  9.217164  76.832457              1    -0.239001   -0.138268    0.066062   

   NDMI_change        cause  
0     0.018204  Urban/Other  
1    -0.088056  Urban/Other  
2    -0.097505  Urban/Other  
3    -0.105232  Urban/Other  
4    -0.044174  Urban/Other  


In [24]:
# Define grid size in degrees (~5 km ≈ 0.045 degrees)
grid_size = 0.045

# Assign each point to a grid cell
df['lat_grid'] = (df['latitude'] // grid_size) * grid_size
df['lon_grid'] = (df['longitude'] // grid_size) * grid_size
# Group by grid and cause
grid_summary = df.groupby(['lat_grid', 'lon_grid', 'cause']).size().reset_index(name='deforestation_count')

# Pivot table to see all causes per grid
grid_pivot = grid_summary.pivot_table(index=['lat_grid','lon_grid'], columns='cause', values='deforestation_count', fill_value=0)

# Optional: total deforestation per grid
grid_pivot['total_deforestation'] = grid_pivot.sum(axis=1)

# Sort by total deforestation descending
top_hotspots = grid_pivot.sort_values(by='total_deforestation', ascending=False).head(10)
print("Top 10 Deforestation Hotspots (Grid-Based):")
print(top_hotspots)


Top 10 Deforestation Hotspots (Grid-Based):
cause              Agriculture  Fire  Logging  Urban/Other  \
lat_grid lon_grid                                            
10.17    76.860          201.0   0.0      2.0        614.0   
         76.815          165.0   0.0      0.0        496.0   
         76.995          323.0   0.0     95.0        182.0   
         76.770          119.0   0.0      3.0        419.0   
         76.725          196.0   0.0      5.0        278.0   
         76.950          180.0   1.0    144.0         99.0   
         77.040           91.0   0.0      2.0        331.0   
         76.905          172.0   0.0     41.0        194.0   
         77.130           82.0   7.0     39.0         87.0   
         77.085           74.0   5.0     11.0         98.0   

cause              total_deforestation  
lat_grid lon_grid                       
10.17    76.860                  817.0  
         76.815                  661.0  
         76.995                  600.0  
      

In [25]:
!pip install geopandas


In [26]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Deforestation_Predictions.csv')

df.head()
import geopandas as gpd
from shapely.geometry import Point

geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]

gdf_points = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"  # WGS84
)


In [27]:
!ls /content/drive/MyDrive



'1-2 Mid Marks Final (1).zip'
'1-2 Mid Marks Final.zip'
 1668518528641674552989016764269.jpg
 21D21A05P6.ipynb
'2-2 Mid Marks (2022 Batch).zip'
 22WH1A05F5
 22WH1A05F5_AI.pdf
 22WH1A05F5_CHAITANYA.pdf
 22WH1A05F5_Cisco.pdf
 22wh1a05f5_CSE_C1.ipynb
'22wh1a05f5-G Chaitanya-Resume (1).pdf'
'22wh1a05f5-G Chaitanya-Resume.pdf'
 22WH1A05F5.gdoc
'22wh1a05f5_Gopagoni Chaitanya.pdf'
 22wh1a05f5_IoT.pdf
 22wh1a05f5_Path_sum.mp4
 22WH1A05F5.pdf
 22WH1A05H3.gdoc
 22WH1A05H3.pdf
'3-1 Mid Marks.zip'
 7e1a28eba37364be1456f9c4688bd2a7-ec3cc1be87ef3f1b6e1e8849ecda78adb40e0f9a.zip
 8mb.video-GUV-wQDY1Gob.mp4
'All States and Districts.json'
 Alstom_Applicant_List_for_Graduate_Engineer_Trainee.xlsx
 AndhraPradesh_Forest_Coordinates.csv
 AndhraPradesh.geojson
 AP_Deforestation_Causes_Adaptive.csv
 AP_Deforestation_Causes_Adaptive_Map.html
 AP_Deforestation_Map.html
 AP_Deforestation_Predictions.csv
 AP_District_Wise_Deforestation_Causes.csv
 AP_District_Wise_Deforestation.csv
 AP_Forest_Coordinates.csv
 ap

In [28]:
import geopandas as gpd

gdf_districts = gpd.read_file(
    '/content/drive/MyDrive/district.geojson'
)

gdf_districts.head()


,DISTRICT,ST_NM,ST_CEN_CD,DT_CEN_CD,censuscode,orig_ogc_f,geometry
0,Alappuzha,Kerala,32,11,598,7,"POLYGON ((76.37334 9.83565, 76.37955 9.82888, ..."
1,Ernakulam,Kerala,32,8,595,173,"POLYGON ((76.68924 10.26721, 76.68724 10.2617,..."
2,Idukki,Kerala,32,9,596,232,"POLYGON ((77.28895 10.22973, 77.29462 10.21643..."
3,Kannur,Kerala,32,2,589,277,"POLYGON ((75.46997 12.30049, 75.48558 12.29131..."
4,Kasaragod,Kerala,32,1,588,288,"POLYGON ((75.41667 12.50166, 75.4224 12.48463,..."


In [29]:
import pandas as pd
from shapely.geometry import Point

df = pd.read_csv('/content/drive/MyDrive/Deforestation_Predictions_kerela.csv')

gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how='left',
    predicate='within'
)

points_with_district.head()
points_with_district.columns
district_deforestation = (
    points_with_district[points_with_district['deforestation'] == 1]
    .groupby('DISTRICT')
    .size()
    .reset_index(name='deforestation_samples')
)

district_deforestation.sort_values(
    'deforestation_samples', ascending=False
)
state_total_deforestation = (
    points_with_district['deforestation'] == 1
).sum()

state_total_deforestation
district_deforestation['state_percentage'] = (
    district_deforestation['deforestation_samples'] /
    state_total_deforestation * 100
)

district_deforestation.sort_values(
    'deforestation_samples', ascending=False
)


,DISTRICT,deforestation_samples,state_percentage
0,Idukki,5111,85.496822
1,Pathanamthitta,867,14.503178


In [30]:
print(gdf_districts.columns)


Index(['DISTRICT', 'ST_NM', 'ST_CEN_CD', 'DT_CEN_CD', 'censuscode',
       'orig_ogc_f', 'geometry'],
      dtype='object')


In [31]:
print(gdf_districts.columns.tolist())

print(gdf_districts.columns.tolist())

['DISTRICT', 'ST_NM', 'ST_CEN_CD', 'DT_CEN_CD', 'censuscode', 'orig_ogc_f', 'geometry']

['DISTRICT', 'ST_NM', 'ST_CEN_CD', 'DT_CEN_CD', 'censuscode', 'orig_ogc_f', 'geometry']
['DISTRICT', 'ST_NM', 'ST_CEN_CD', 'DT_CEN_CD', 'censuscode', 'orig_ogc_f', 'geometry']


['DISTRICT',
 'ST_NM',
 'ST_CEN_CD',
 'DT_CEN_CD',
 'censuscode',
 'orig_ogc_f',
 'geometry']

In [32]:
import pandas as pd
import geopandas as gpd
import folium


In [33]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np


# =====================================================
# STEP 2: Load Kerala District Boundary
# =====================================================
gdf_districts = gpd.read_file(
    "/content/drive/MyDrive/district.geojson"
)

if gdf_districts.crs is None:
    gdf_districts.set_crs(epsg=4326, inplace=True)

gdf_districts = gdf_districts.to_crs(epsg=4326)
gdf_districts["state"] = "Kerala"


# =====================================================
# STEP 3: Load Deforestation Prediction CSV
# =====================================================
df = pd.read_csv(
    "/content/drive/MyDrive/Deforestation_Predictions_kerela.csv"
)

# Ensure afforestation column exists
if "afforestation" not in df.columns:
    df["afforestation"] = 0


# =====================================================
# STEP 4: Convert Prediction Points to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)


# =====================================================
# STEP 5: Spatial Join (Points → Districts)
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)


# =====================================================
# STEP 6: District-wise Sample Counts
# =====================================================
district_total = (
    points_with_district
    .groupby("DISTRICT")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("DISTRICT")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("DISTRICT")
    .size()
    .reset_index(name="afforestation_samples")
)


# =====================================================
# STEP 7: Merge Statistics with District Map
# =====================================================
gdf_districts = gdf_districts.merge(
    district_total, on="DISTRICT", how="left"
)
gdf_districts = gdf_districts.merge(
    district_deforestation, on="DISTRICT", how="left"
)
gdf_districts = gdf_districts.merge(
    district_afforestation, on="DISTRICT", how="left"
)

gdf_districts.fillna(0, inplace=True)


# =====================================================
# STEP 8: Calculate Rates & Area
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

# Deforestation
gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

# Afforestation
gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)


# =====================================================
# STEP 9: Create Interactive Folium Map
# =====================================================
m = folium.Map(location=[10.85, 76.27], zoom_start=7)


# =====================================================
# STEP 10: Dynamic Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, 6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["DISTRICT", "deforestation_samples"],
    key_on="feature.properties.DISTRICT",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Kerala)"
).add_to(m)


# =====================================================
# STEP 11: District Tooltip
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "DISTRICT",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation area (sq.km):",
            "Deforestation area (hectares):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation area (sq.km):",
            "Afforestation area (hectares):"
        ],
        localize=True,
        sticky=True
    )
).add_to(m)


# =====================================================
# STEP 12: Kerala Forest Alert Popup
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(
        by="deforestation_samples",
        ascending=False
    ).head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['DISTRICT']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Kerala Forest Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Recommended Actions:<br>
• Western Ghats protection monitoring<br>
• Community forest conservation<br>
• Mangrove restoration<br>
• Strict land-use regulation
"""

folium.Marker(
    location=[10.85, 76.27],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)


# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl(collapsed=False).add_to(m)

m.save(
    "/content/drive/MyDrive/Kerala_Forest_Monitoring_Map.html"
)

print("✅ Kerala forest monitoring map saved successfully!")

✅ Kerala forest monitoring map saved successfully!


In [34]:
points_with_district.groupby('DISTRICT')['deforestation'].mean().sort_values(ascending=False)


,deforestation
DISTRICT,
Pathanamthitta,0.997699
Idukki,0.996102
